# TOOLS
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    1. Aschema, including the name of the tool, a descirption, and/or argument definitions(often a JSON schema)
    2. A function or coroutine to execute

In [9]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-20b", model_provider="groq")
response = model.invoke("why is there gravity on earth?")
print(response.content)

**Short answer:**  
Because Earth has mass (and, in Einstein’s language, energy). Mass tells spacetime how to curve, and that curvature makes objects “fall” toward Earth. The result is the familiar 9.8 m s⁻² pull we call gravity.

---

## 1. The Newtonian view  
- **Gravity is a force:** Sir Isaac Newton described it with his law of universal gravitation:  
  \[
  F = G \frac{M_{\text{Earth}}\;m}{r^{2}}
  \]
  where \(M_{\text{Earth}}\) is Earth’s mass, \(m\) is the mass of the other object, \(r\) is the distance between their centers, and \(G\) is the gravitational constant.  
- **Why it exists:** Any object with mass attracts every other mass. Earth, being a huge sphere of matter, pulls everything toward its center.  
- **Result:** Objects accelerate downward at ~\(9.8\ \text{m/s}^{2}\) (on average) because that’s the strength of Earth’s pull at its surface.

## 2. The Einsteinian view  
- **Gravity is geometry:** Einstein’s general relativity says mass–energy curves the fabric of sp

In [10]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [12]:
response = model_with_tools.invoke("whats the weather like in boston?")
print(response.content)
for tool_call in response.tool_calls:
    #view tool calls made by the model

    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


Tool: get_weather
Args: {'location': 'Boston'}


## tool execution loops


In [15]:
#step 1: model generated tool calls
messages = [{"role": "user", "content": "What's the weather in boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3: pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)


It’s sunny in Boston! ☀️
